# Digital-Forensics File-Fragment Intelligence Module
## Byte2Image + Swin Transformer V2 (Tiny) Training on FFT-75

This notebook is designed for the Google Colab GPU runtime.

### Pipeline

1. Raw **512-byte fragment**
2. Published **Byte2Image** representation
   - 1-bit sliding-byte window
   - intra-byte n-grams
   - native **497×128 grayscale**
3. Resize native representation to **256×256** for Swin V2 Tiny
4. ImageNet-pretrained **Swin Transformer V2 Tiny**
5. 75-class classification
6. Staged training and evaluation

### Important

The native Byte2Image representation is **not** a 256×256 transition matrix.

For 512 bytes and `n=16`:

- shifted byte matrix: `512×8`
- native Byte2Image: `497×128`
- `256×256`: downstream model-input adaptation only

This notebook does not overwrite the real FFT-75 dataset loader with synthetic magic-signature data.


In [1]:
# =============================================================================
# CELL 1: Environment & GPU Diagnostics
# =============================================================================
import os
import sys
import platform
import torch

print("=" * 70)
print("GOOGLE COLAB HARDWARE & ENVIRONMENT DIAGNOSTICS")
print("=" * 70)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Platform:         {platform.platform()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print("CUDA Available:   YES")
    print(f"CUDA Version:     {torch.version.cuda}")
    print(f"GPU Model:        {gpu_name}")
    print(f"Total VRAM:       {vram_gb:.2f} GB")
    rec_batch = 64 if vram_gb >= 14 else 32 if vram_gb >= 8 else 16
    print(f"Initial Batch Recommendation: {rec_batch}")
else:
    print("CUDA Available:   NO")
    print("WARNING: Training will run on CPU.")
print("=" * 70)


GOOGLE COLAB HARDWARE & ENVIRONMENT DIAGNOSTICS
Python Version:   3.13.4
PyTorch Version:  2.13.0+cpu
Platform:         Windows-11-10.0.26200-SP0
CUDA Available:   NO


In [2]:
# =============================================================================
# CELL 2: Unified Repository Workspace Setup
# =============================================================================
import os
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Cleanly clone fresh repository in Colab
if os.path.exists("/content"):
    repo_path = Path("/content/SIH-FirSeFile")
    if repo_path.exists():
        print("Removing old cached repo directory...")
        shutil.rmtree(str(repo_path), ignore_errors=True)
    print("Cloning latest SIH-FirSeFile from GitHub...")
    subprocess.run(["git", "clone", "https://github.com/kusha-l921/SIH-FirSeFile.git", "/content/SIH-FirSeFile"], check=True)
    PROJECT_ROOT = repo_path
else:
    PROJECT_ROOT = Path(os.path.abspath("."))

# 2. Set working directory and sys.path
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. Install required packages
!pip install -q timm transformers scikit-learn pandas seaborn matplotlib pyyaml networkx pillow

print(f"\nCanonical PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Working Directory:      {os.getcwd()}")
print(f"Source modules:         {sorted(os.listdir(PROJECT_ROOT / 'src'))}")
print(f"Configs available:      {sorted(os.listdir(PROJECT_ROOT / 'configs'))}")


'rm' is not recognized as an internal or external command,
operable program or batch file.


fatal: destination path '/content/SIH-FirSeFile' already exists and is not an empty directory.



Canonical PROJECT_ROOT: \content\SIH-FirSeFile
Working Directory:      C:\content\SIH-FirSeFile
Source modules:         ['__init__.py', '__pycache__', 'baselines', 'datasets', 'models', 'reassembly', 'representations', 'training', 'utils', 'validation']
Configs available:      ['main_run.yaml', 'pilot_run.yaml', 'smoke_test.yaml']



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# =============================================================================
# CELL 3: Backup Directory Setup
# =============================================================================
from src.utils.colab_utils import setup_backup_directory

# Set use_google_drive=True to mount Google Drive for checkpoint backup
backup_dir = setup_backup_directory(use_google_drive=False)
print(f"Checkpoint Backup Directory: {backup_dir}")


Using local backup directory: C:\content\SIH-FirSeFile\local_backups
Checkpoint Backup Directory: C:\content\SIH-FirSeFile\local_backups


In [4]:
# =============================================================================
# CELL 4: Import Pure Repository Byte2Image Module
# =============================================================================
from src.representations.byte2image import (
    sliding_byte_window,
    byte2image_native,
    bytes_to_byte2image,
    Byte2ImageTransform
)

print("=" * 70)
print("BYTE2IMAGE MODULE IMPORTED FROM REPOSITORY")
print("=" * 70)
print("Fragment size : 512 bytes")
print("n-gram        : 16")
print("Native image  : 497 x 128 (Grayscale)")
print("Swin input    : 256 x 256 (Model-Input Adaptation)")
print("Channels      : 1")
print("=" * 70)


BYTE2IMAGE MODULE IMPORTED FROM REPOSITORY
Fragment size : 512 bytes
n-gram        : 16
Native image  : 497 x 128 (Grayscale)
Swin input    : 256 x 256 (Model-Input Adaptation)
Channels      : 1


In [5]:
# =============================================================================
# CELL 5: Byte2Image Research & Numerical Verification
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from src.representations.byte2image import (
    sliding_byte_window,
    byte2image_native,
    bytes_to_byte2image
)

print("=" * 70)
print("BYTE2IMAGE RESEARCH VERIFICATION")
print("=" * 70)

# TEST 1: Sliding-byte construction
test_fragment = bytes([0xFF, 0xD8, 0xFF, 0xE0]) + bytes(508)
shifted = sliding_byte_window(test_fragment)
expected_4x4 = np.array([
    [0xFF, 0xFF, 0xFF, 0xFE],
    [0xD8, 0xB1, 0x63, 0xC7],
    [0xFF, 0xFF, 0xFF, 0xFF],
    [0xE0, 0xC0, 0x80, 0x00],
], dtype=np.uint8)

print("\n[TEST 1] Shifted matrix:", shifted.shape)
assert shifted.shape == (512, 8), f"Expected (512, 8), got {shifted.shape}"
assert np.array_equal(shifted[:4, :4], expected_4x4), "Shifted 4x4 matrix mismatch"
print("[PASS] Sliding-byte construction verified.")

# TEST 2: Native Byte2Image dimensions
native = byte2image_native(test_fragment, ngram=16)
print("\n[TEST 2] Native shape:", native.shape)
assert native.shape == (497, 128), f"Expected (497, 128), got {native.shape}"
assert native.dtype == np.uint8
print("[PASS] Native dimensions verified (497 x 128).")

# TEST 3: Realistic non-trivial byte distribution
rng = np.random.default_rng(42)
realistic_fragment = rng.integers(0, 256, size=512, dtype=np.uint8).tobytes()
realistic_native = byte2image_native(realistic_fragment, ngram=16)
realistic_swin = bytes_to_byte2image(realistic_fragment, target_size=(256, 256), ngram=16, normalize=False)

print("\n[TEST 3] Realistic Fragment Statistics:")
print(f"  Mean: {realistic_native.mean():.2f} | Std: {realistic_native.std():.2f}")
print(f"  Unique values: {len(np.unique(realistic_native))} | Non-zero: {np.mean(realistic_native != 0)*100:.2f}%")
assert realistic_native.std() > 0 and len(np.unique(realistic_native)) > 10
print("[PASS] Realistic fragment produces rich non-trivial 2D texture.")

# TEST 4: Swin 256 x 256 Model-Input Adaptation
print("\n[TEST 4] Swin image shape:", realistic_swin.shape)
assert realistic_swin.shape == (256, 256)
print("[PASS] 256 x 256 Swin input adaptation verified.")

# TEST 5: Determinism
assert np.array_equal(byte2image_native(realistic_fragment), byte2image_native(realistic_fragment))
print("\n[TEST 5] [PASS] Transformation is 100% deterministic.")

# TEST 6: Inter-Fragment Differentiation
second_fragment = rng.integers(0, 256, size=512, dtype=np.uint8).tobytes()
diff_rate = np.mean(realistic_native != byte2image_native(second_fragment))
print(f"\n[TEST 6] Inter-fragment difference rate: {diff_rate*100:.2f}%")
assert diff_rate > 0.5
print("[PASS] Different fragments produce distinct 2D representations.")
print("=" * 70)
print("BYTE2IMAGE RESEARCH VERIFICATION COMPLETE")
print("=" * 70)


BYTE2IMAGE RESEARCH VERIFICATION

[TEST 1] Shifted matrix: (512, 8)
[PASS] Sliding-byte construction verified.

[TEST 2] Native shape: (497, 128)
[PASS] Native dimensions verified (497 x 128).

[TEST 3] Realistic Fragment Statistics:
  Mean: 127.43 | Std: 73.05
  Unique values: 256 | Non-zero: 99.72%
[PASS] Realistic fragment produces rich non-trivial 2D texture.

[TEST 4] Swin image shape: (256, 256)
[PASS] 256 x 256 Swin input adaptation verified.

[TEST 5] [PASS] Transformation is 100% deterministic.

[TEST 6] Inter-fragment difference rate: 99.52%
[PASS] Different fragments produce distinct 2D representations.
BYTE2IMAGE RESEARCH VERIFICATION COMPLETE


In [6]:
# =============================================================================
# CELL 6: Training-Pipeline Integration Audit
# =============================================================================
import os
import sys
import importlib
from pathlib import Path

# Candidate paths to locate project root
candidates = [
    Path("/content/SIH-FirSeFile"),
    Path("/content"),
    Path(os.path.abspath(".")),
]

detected_root = None
for c in candidates:
    if (c / "src" / "training" / "train.py").exists() and (c / "configs" / "smoke_test.yaml").exists():
        detected_root = c
        break

if detected_root is None:
    raise FileNotFoundError(
        f"Could not find project root containing src/training/train.py. Candidates checked: {candidates}\n"
        "Please make sure you ran Cell 2 to clone the repository."
    )

PROJECT_ROOT = detected_root
os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_paths = [
    PROJECT_ROOT / "src" / "training" / "train.py",
    PROJECT_ROOT / "src" / "models" / "swin_v2.py",
    PROJECT_ROOT / "src" / "datasets" / "fft75.py",
    PROJECT_ROOT / "src" / "datasets" / "fragment_dataset.py",
    PROJECT_ROOT / "configs" / "smoke_test.yaml",
    PROJECT_ROOT / "configs" / "pilot_run.yaml",
    PROJECT_ROOT / "configs" / "main_run.yaml",
]

print("=" * 70)
print("TRAINING PIPELINE INTEGRATION AUDIT")
print("=" * 70)
print(f"Auditing canonical PROJECT_ROOT: {PROJECT_ROOT}")

for path in required_paths:
    exists = path.exists()
    print(f"{'OK  ' if exists else 'MISS'} {path.relative_to(PROJECT_ROOT)}")
    if not exists:
        raise FileNotFoundError(f"Missing required path: {path}")

# Verify Byte2Image representation module
module = importlib.import_module("src.representations.byte2image")
print("\nLoaded representation module:", module.__file__)
assert hasattr(module, "bytes_to_byte2image"), "Missing bytes_to_byte2image"
assert hasattr(module, "byte2image_native"), "Missing byte2image_native"
assert hasattr(module, "sliding_byte_window"), "Missing sliding_byte_window"
print("[PASS] Correct Byte2Image module is importable with all required APIs.")

# Verify Swin Transformer V2 model
swin_module = importlib.import_module("src.models.swin_v2")
print("Swin model module:            ", swin_module.__file__)
assert hasattr(swin_module, "SwinV2TinyGrayscale"), "Missing SwinV2TinyGrayscale"
print("[PASS] SwinV2TinyGrayscale model class is available.")

# Verify Training pipeline
train_module = importlib.import_module("src.training.train")
print("Training pipeline module:     ", train_module.__file__)
assert hasattr(train_module, "run_staged_training"), "Missing run_staged_training"
print("[PASS] run_staged_training function is available.")

print("\n" + "=" * 70)
print("INTEGRATION AUDIT COMPLETE — READY FOR GPU TRAINING")
print("=" * 70)


TRAINING PIPELINE INTEGRATION AUDIT
Auditing canonical PROJECT_ROOT: \content\SIH-FirSeFile
OK   src\training\train.py
OK   src\models\swin_v2.py
OK   src\datasets\fft75.py
OK   src\datasets\fragment_dataset.py
OK   configs\smoke_test.yaml
OK   configs\pilot_run.yaml
OK   configs\main_run.yaml

Loaded representation module: C:\content\SIH-FirSeFile\src\representations\byte2image.py
[PASS] Correct Byte2Image module is importable with all required APIs.


C:\Users\bhavy\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Swin model module:             C:\content\SIH-FirSeFile\src\models\swin_v2.py
[PASS] SwinV2TinyGrayscale model class is available.


Training pipeline module:      C:\content\SIH-FirSeFile\src\training\train.py
[PASS] run_staged_training function is available.

INTEGRATION AUDIT COMPLETE — READY FOR GPU TRAINING


In [ ]:
# =============================================================================
# CELL 7: Phase 0 — Smoke Test (100 samples/class, 1 epoch)
# =============================================================================
import yaml
from src.training.train import run_staged_training

smoke_config_path = str(PROJECT_ROOT / "configs" / "smoke_test.yaml")
with open(smoke_config_path, "r") as f:
    smoke_config = yaml.safe_load(f)

print("=" * 70)
print("PHASE 0 — SMOKE TEST EXECUTION")
print("=" * 70)

smoke_result = run_staged_training(smoke_config)
print("\n[OK] Phase 0 Smoke Test completed successfully.")
if isinstance(smoke_result, dict):
    print("Result keys:", list(smoke_result.keys()))


In [ ]:
# =============================================================================
# CELL 8: Phase 1 — Pilot Run (1,000 samples/class, ~75k samples)
# =============================================================================
import yaml
from src.training.train import run_staged_training

pilot_config_path = str(PROJECT_ROOT / "configs" / "pilot_run.yaml")
with open(pilot_config_path, "r") as f:
    pilot_config = yaml.safe_load(f)

print("=" * 70)
print("PHASE 1 — PILOT RUN EXECUTION")
print("=" * 70)

pilot_result = run_staged_training(pilot_config, drive_backup_dir=backup_dir)
print("\n[OK] Phase 1 Pilot Run completed successfully.")
if isinstance(pilot_result, dict) and "best_val_macro_f1" in pilot_result:
    print(f"Best Validation Macro-F1: {pilot_result['best_val_macro_f1']:.4f}")


In [ ]:
# =============================================================================
# CELL 9: Phase 2 — Main Fine-Tuning Run (2,000 samples/class, ~150k samples)
# =============================================================================
import yaml
from src.training.train import run_staged_training

main_config_path = str(PROJECT_ROOT / "configs" / "main_run.yaml")
with open(main_config_path, "r") as f:
    main_config = yaml.safe_load(f)

print("=" * 70)
print("PHASE 2 — MAIN FINE-TUNING RUN EXECUTION")
print("=" * 70)

main_result = run_staged_training(main_config, drive_backup_dir=backup_dir)
print("\n[OK] Phase 2 Main Run completed successfully.")
if isinstance(main_result, dict) and "best_val_macro_f1" in main_result:
    print(f"Best Validation Macro-F1: {main_result['best_val_macro_f1']:.4f}")


In [ ]:
# =============================================================================
# CELL 10: Baseline Comparison Table
# =============================================================================

from src.baselines.byteformer import (
    create_comparison_table
)

measured_metrics = None

if isinstance(main_result, dict):
    measured_metrics = main_result.get("test_metrics")

if measured_metrics is None and isinstance(pilot_result, dict):
    measured_metrics = pilot_result.get("test_metrics")

if measured_metrics is None and isinstance(smoke_result, dict):
    measured_metrics = smoke_result.get("test_metrics")

comparison_df = create_comparison_table(
    our_measured_metrics=measured_metrics
)

print("\n=======================================================")
print("FFT-75 FILE-FRAGMENT BENCHMARK COMPARISON")
print("=======================================================")
print(comparison_df.to_string(index=False))
print("=======================================================")


In [ ]:
# =============================================================================
# CELL 11: Display Training History & Saved Visualizations
# =============================================================================

from IPython.display import Image, display
import glob

plot_images = sorted(
    glob.glob(
        "experiments/**/*.png",
        recursive=True
    )
)

if not plot_images:
    print("No experiment PNG files were found yet.")
else:
    print(f"Found {len(plot_images)} saved plots.")

    for path in plot_images:
        print(f"\nPlot: {path}")
        display(Image(filename=path))


## Final experimental record

Record the following after the runs complete.

### Representation
- Native Byte2Image: **497×128**
- Swin input adaptation: **256×256**
- Input channels: **1**
- n-gram: **16**
- Fragment size: **512 bytes**

### Model
- Swin Transformer V2 Tiny
- ImageNet-pretrained initialization
- grayscale input adaptation
- 75 output classes

### Evaluation
- Top-1 accuracy
- Top-5 accuracy
- Macro-F1
- per-class metrics
- confusion matrix
- inference throughput/latency where measured

### Research integrity
- Published literature values remain labeled as literature.
- Experimental measurements remain labeled as our measurements.
- Do not claim that the original Byte2Image paper used Swin V2.
- The experimental contribution is the use of the published Byte2Image representation with the chosen vision-transformer classifier in the proposed forensic recovery pipeline.

### Reproducibility
Save:
- best checkpoint
- final checkpoint
- configuration files
- random seeds
- split information
- class mapping
- metrics
- training curves
- environment/GPU information
